In [1]:
from pathlib import Path
import sys

import torch
from omegaconf import OmegaConf

REPO_ROOT = Path("/dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni")
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))
print(REPO_ROOT)

from omni_speech.backbone_trainer import BackboneTrainingModule, TextDataModule
from omni_speech.constants import IGNORE_INDEX
from omni_speech.conversation import default_conversation

/dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni/.venv-a100/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni


/dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni/.venv-a100/lib/python3.11/site-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni/.venv-a100/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni/.venv-a100/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
CONFIG_PATH = REPO_ROOT / "configs" / "backbone.yaml"
APPLY_LORA = True  # Set True if you want this notebook to mirror LoRA wrapping exactly.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = OmegaConf.load(CONFIG_PATH)
if "hydra" in cfg:
    del cfg["hydra"]

# The trainer resolves these relative paths from the repo root via Hydra.
# In a notebook, the kernel cwd can differ, so make the same paths explicit.
cfg.data.path = str(REPO_ROOT / "data" / "instruct")
cfg.data.json_path = str(REPO_ROOT / "data" / "instruct" / "hindi_instruct_conversations.json")
cfg.model.path = str(REPO_ROOT / "models" / "llama")
cfg.model.config_path = str(REPO_ROOT / "models" / "llama")
cfg.model.model_base = str(REPO_ROOT / "models" / "llama")
cfg.model.tokenizer_path = str(REPO_ROOT / "models" / "llama")

cfg.data.num_workers = 0
cfg.training.use_lora = APPLY_LORA
cfg.training.gradient_checkpointing = False
cfg.training.devices = 1
cfg.training.accelerator = "gpu" if DEVICE == "cuda" else "cpu"

In [3]:
import subprocess
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Fri_Jun_14_16:34:21_PDT_2024
Cuda compilation tools, release 12.6, V12.6.20
Build cuda_12.6.r12.6/compiler.34431801_0



In [4]:
import torch, os

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(os.environ.get("CUDA_VISIBLE_DEVICES"))

True
1
MIG-caff2386-f378-5dd6-bbd2-45fccf1e9379


In [5]:
module = BackboneTrainingModule(cfg)
module = module.half()
module = module.to(DEVICE)
module.eval()

tokenizer = module.tokenizer
model = module.model

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.17s/it]
Some weights of the model checkpoint at /dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni/models/llama were not used when initializing OmniSpeechLlamaForCausalLM: ['speech_generator.input_proj.bias', 'speech_generator.input_proj.weight', 'speech_generator.layers.0.input_layernorm.weight', 'speech_generator.layers.0.mlp.down_proj.weight', 'speech_generator.layers.0.mlp.gate_proj.weight', 'speech_generator.layers.0.mlp.up_proj.weight', 'speech_generator.layers.0.post_attention_layernorm.weight', 'speech_generator.layers.0.self_attn.k_proj.weight', 'speech_generator.layers.0.self_attn.o_proj.weight', 'speech_generator.layers.0.self_attn.q_proj.weight', 'speech_generator.layers.0.self_attn.v_proj.weight', 'speech_generator.layers.1.input_layernorm.weight', 'speech_generator.layers.1.mlp.down_proj.weight', 'speech_generator.layers.1.mlp.gate_proj.weight', 'speech_generator.layers.1.mlp.up_proj.weight', 'sp

trainable params: 83,886,080 || all params: 8,770,698,240 || trainable%: 0.9564
Trainable parameters: 83,886,080 / 8,770,698,240 (0.96%) [projector=False, llm=True, lora=True, encoder=False]


In [12]:
data_module = TextDataModule(cfg, tokenizer, model.config)
data_module.setup("fit")
train_loader = data_module.train_dataloader()

Loaded 102099 text-only samples from /dss/dsshome1/0C/ra85muk2/Desktop/Programming/hindi_llama_omni/data/instruct/hindi_instruct_conversations.json (skipped 0 malformed samples).


In [15]:
# This is the same batch construction path used by Trainer.fit(...):
# TextDataModule.train_dataloader() -> TextLengthBucketSampler -> TextCollator.
padded_batch = next(iter(train_loader))
single_batch = {k: v[:1] for k, v in padded_batch.items()}

single_batch = {k: v.to(DEVICE) for k, v in single_batch.items()}
padded_batch = {k: v.to(DEVICE) for k, v in padded_batch.items()}

print("=" * 100)
print("Batch from actual training dataloader")
print("batch size:", padded_batch["input_ids"].shape[0])
print("sequence length:", padded_batch["input_ids"].shape[1])
print("pad counts per row:", (padded_batch["input_ids"] == tokenizer.pad_token_id).sum(dim=1).detach().cpu().tolist())
print("ignore-index counts per row:", (padded_batch["labels"] == IGNORE_INDEX).sum(dim=1).detach().cpu().tolist())

Batch from actual training dataloader
batch size: 1
sequence length: 184
pad counts per row: [0]
ignore-index counts per row: [100]


In [16]:
print("\nDecoded first dataloader sample:")
print(tokenizer.decode(padded_batch["input_ids"][0].detach().cpu().tolist(), skip_special_tokens=False))


Decoded first dataloader sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

आप एक सहायक भाषा और वाणी सहायक हैं। उपयोगकर्ता की बात समझकर प्राकृतिक भाषा में सहायता करें।<|eot_id|><|start_header_id|>user<|end_header_id|>

+ सीज़न 1 और सीज़न 2 में क्या अंतर था? अगर यह प्रतिक्रिया है, तो पहले क्या आया था?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

रेनो रंबल (सीज़न 2) रेनो रंबलः ईस्ट बनाम वेस्ट ऑस्ट्रेलियाई रियलिटी टेलीविजन सीरीज़ "रेनो रंबल" का दूसरा सीज़न है, जो नाइन नेटवर्क पर प्रसारित होता है।<|eot_id|>


In [18]:
print("\n" + "=" * 100)
print("Single-sample tensors")
print("input_ids shape:", tuple(padded_batch["input_ids"].shape))
print("labels shape:", tuple(padded_batch["labels"].shape))
print("attention_mask shape:", tuple(padded_batch["attention_mask"].shape))
print("x (input_ids):")
print(padded_batch["input_ids"][0].detach().cpu().tolist())


Single-sample tensors
input_ids shape: (1, 184)
labels shape: (1, 184)
attention_mask shape: (1, 184)
x (input_ids):
[128000, 128006, 9125, 128007, 271, 120470, 100549, 103887, 109953, 100348, 102580, 24810, 100358, 100287, 103660, 44747, 103887, 109953, 85410, 101130, 106261, 100666, 101185, 100365, 24810, 48909, 44747, 100276, 100444, 108508, 101185, 84736, 86133, 101315, 102861, 100349, 100348, 102580, 24810, 92317, 100271, 103887, 110239, 24810, 100331, 109290, 128009, 128006, 882, 128007, 271, 10, 69258, 102882, 109140, 220, 16, 100358, 69258, 102882, 109140, 220, 17, 92317, 100271, 48909, 100305, 24810, 100285, 108246, 100518, 24810, 30, 108858, 100866, 84736, 101922, 100349, 86133, 100322, 24810, 85410, 101411, 100329, 55675, 102348, 35470, 48909, 100305, 24810, 103813, 24810, 100518, 24810, 30, 128009, 128006, 78191, 128007, 271, 45279, 101276, 55675, 100303, 102829, 92911, 320, 79468, 102882, 109140, 220, 17, 8, 100303, 101276, 55675, 100303, 102829, 92911, 111166, 106512, 79

In [19]:


print("\ny (raw labels, including IGNORE_INDEX where present):")
print(padded_batch["labels"][0].detach().cpu().tolist())


y (raw labels, including IGNORE_INDEX where present):
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 45279, 101276, 55675, 100303, 102829, 92911, 320, 79468, 102882, 109140, 220, 17, 8, 100303, 101276, 55675, 100303, 102829, 92911, 111166, 106512, 79468, 100431, 101403, 100497, 100287, 101755, 100431, 103630, 79468, 100431, 86133, 101385, 100322, 101344, 100303, 121740, 102399, 44747, 101002, 101385, 102646, 122879, 69258, 101815, 102882, 10

In [20]:

input_row = padded_batch["input_ids"][0].detach().cpu()
label_row = padded_batch["labels"][0].detach().cpu()
ignored_input_ids = input_row[label_row == IGNORE_INDEX]
trained_label_ids = label_row[label_row != IGNORE_INDEX]

print("\nDecoded ignored label span (system + user prompt + assistant header, plus any label padding):")
print(tokenizer.decode(ignored_input_ids.tolist(), skip_special_tokens=False))
print("\nDecoded trained label span (assistant answer tokens used for loss):")
print(tokenizer.decode(trained_label_ids.tolist(), skip_special_tokens=False))
print("\nDecoded full training input:")
print(tokenizer.decode(input_row.tolist(), skip_special_tokens=False))



Decoded ignored label span (system + user prompt + assistant header, plus any label padding):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

आप एक सहायक भाषा और वाणी सहायक हैं। उपयोगकर्ता की बात समझकर प्राकृतिक भाषा में सहायता करें।<|eot_id|><|start_header_id|>user<|end_header_id|>

+ सीज़न 1 और सीज़न 2 में क्या अंतर था? अगर यह प्रतिक्रिया है, तो पहले क्या आया था?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Decoded trained label span (assistant answer tokens used for loss):
रेनो रंबल (सीज़न 2) रेनो रंबलः ईस्ट बनाम वेस्ट ऑस्ट्रेलियाई रियलिटी टेलीविजन सीरीज़ "रेनो रंबल" का दूसरा सीज़न है, जो नाइन नेटवर्क पर प्रसारित होता है।<|eot_id|>

Decoded full training input:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

आप एक सहायक भाषा और वाणी सहायक हैं। उपयोगकर्ता की बात समझकर प्राकृतिक भाषा में सहायता करें।<|eot_id|><|start_header_id|>user<|end_header_id|>

+ सीज़न 1 और सीज़न 2 में क्या अंतर था? अगर यह प्रतिक्रिया है, तो पहले क्या आया था?<|eot_id|><|st

In [21]:
import os, torch

print("host:", os.uname().nodename)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("cuda:", torch.cuda.is_available())
print("count:", torch.cuda.device_count())
print("name:", torch.cuda.get_device_name(0))

x = torch.randn(2, 10, device="cuda")
y = torch.tensor([1, 2], device="cuda")
print(torch.nn.functional.cross_entropy(x, y))

host: lrz-dgx-a100-003
CUDA_VISIBLE_DEVICES: MIG-caff2386-f378-5dd6-bbd2-45fccf1e9379
cuda: True
count: 1
name: NVIDIA A100-SXM4-40GB MIG 3g.20gb
tensor(3.3372, device='cuda:0')


In [22]:
with torch.no_grad():
    single_outputs = model(
        input_ids=single_batch["input_ids"],
        attention_mask=single_batch["attention_mask"],
        labels=single_batch["labels"],
        speech=None,
        speech_lengths=None,
        use_cache=False,
        output_hidden_states=True,
        return_dict=True,
    )

In [23]:
    padded_outputs = model(
        input_ids=padded_batch["input_ids"],
        attention_mask=padded_batch["attention_mask"],
        labels=padded_batch["labels"],
        speech=None,
        speech_lengths=None,
        use_cache=False,
        output_hidden_states=True,
        return_dict=True,
    )

In [26]:

print("\n" + "=" * 100)
print("Padded 2-sample batch tensors")
print("input_ids shape:", tuple(padded_batch["input_ids"].shape))
print("labels shape:", tuple(padded_batch["labels"].shape))
print("attention_mask shape:", tuple(padded_batch["attention_mask"].shape))
print("pad_token_id:", tokenizer.pad_token_id)
print("IGNORE_INDEX:", IGNORE_INDEX)
print("pad counts per row:", (padded_batch["input_ids"] == tokenizer.pad_token_id).sum(dim=1).detach().cpu().tolist())
print("ignore-index counts per row:", (padded_batch["labels"] == IGNORE_INDEX).sum(dim=1).detach().cpu().tolist())
print("\nattention_mask rows:")
print(padded_batch["attention_mask"].detach().cpu().int())
print("\nFirst row raw labels:")
print(padded_batch["labels"][0].detach().cpu().tolist())
print("\nSecond row raw labels:")
print(padded_batch["labels"][0].detach().cpu().tolist())



Padded 2-sample batch tensors
input_ids shape: (1, 184)
labels shape: (1, 184)
attention_mask shape: (1, 184)
pad_token_id: 128004
IGNORE_INDEX: -100
pad counts per row: [0]
ignore-index counts per row: [100]

attention_mask rows:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=torch.int32)

First row raw labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

In [27]:

print("\n" + "=" * 100)
print("Forward pass outputs from the model's built-in loss path")
print("Single-row view logits shape:", tuple(single_outputs.logits.shape))

print("Single-row view hidden-state shape:", tuple(single_outputs.hidden_states[-1].shape))
print("Single-row view loss:", float(single_outputs.loss.item()))
print("\nTraining dataloader batch logits shape:", tuple(padded_outputs.logits.shape))
print("Training dataloader batch hidden-state shape:", tuple(padded_outputs.hidden_states[-1].shape))
print("Training dataloader batch loss:", float(padded_outputs.loss.item()))

single_argmax = single_outputs.logits.argmax(dim=-1)[0].detach().cpu().tolist()
print("\nTeacher-forced argmax token ids for the first dataloader row:")
print(single_argmax)
print("\nDecoded teacher-forced argmax sequence:")
print(tokenizer.decode(single_argmax, skip_special_tokens=False))

print("\nFirst 5 token vectors from the final hidden state (shape [5, hidden_size]):")
print(single_outputs.hidden_states[-1][0, :5, :8].detach().cpu())

print("\nDone: batches now come from TextDataModule.train_dataloader(), and loss is computed through model(..., labels=...) like training.")


Forward pass outputs from the model's built-in loss path
Single-row view logits shape: (1, 184, 128256)
Single-row view hidden-state shape: (1, 184, 4096)
Single-row view loss: 1.704706072807312

Training dataloader batch logits shape: (1, 184, 128256)
Training dataloader batch hidden-state shape: (1, 184, 4096)
Training dataloader batch loss: 1.704706072807312

Teacher-forced argmax token ids for the first dataloader row:
[128006, 128006, 198, 128006, 791, 65804, 69258, 100314, 15592, 105542, 24810, 92317, 126388, 101043, 102350, 48909, 109953, 48909, 100400, 128009, 100666, 101185, 100365, 24810, 48909, 35470, 110421, 100444, 100311, 61196, 106261, 86133, 100625, 102861, 100349, 102302, 102580, 24810, 92317, 100271, 116203, 44747, 24810, 84736, 101287, 128009, 128006, 78191, 128009, 78191, 65804, 17, 100460, 109140, 220, 18, 11, 220, 102882, 109140, 220, 17, 48909, 100271, 48909, 101241, 24810, 100285, 108246, 85410, 24810, 30, 128009, 100991, 100549, 107949, 101795, 86133, 100322, 

In [29]:
import torch.nn.functional as F

row_idx = 0  # second row

logits = padded_outputs.logits[row_idx]      # [seq_len, vocab_size]
labels = padded_batch["labels"][row_idx]     # [seq_len]

shift_logits = logits[:-1, :].contiguous()
shift_labels = labels[1:].contiguous()

loss = F.cross_entropy(
    shift_logits,
    shift_labels,
    ignore_index=IGNORE_INDEX,
)

print(loss.item())

1.704706072807312


In [30]:
pred_ids = single_outputs.logits.argmax(dim=-1)[0].detach().cpu()
labels = single_batch["labels"][0].detach().cpu()

shift_pred_ids = pred_ids[:-1]
shift_labels = labels[1:]

mask = shift_labels != IGNORE_INDEX

print("PREDICTIONS USED FOR LOSS:")
print(shift_pred_ids[mask].tolist())

print("\nTRUE LABELS:")
print(shift_labels[mask].tolist())

print("\nDecoded predictions used for loss:")
print(tokenizer.decode(shift_pred_ids[mask].tolist(), skip_special_tokens=False))

print("\nDecoded true labels:")
print(tokenizer.decode(shift_labels[mask].tolist(), skip_special_tokens=False))

PREDICTIONS USED FOR LOSS:
[79468, 101153, 45279, 48909, 101993, 92911, 69258, 35725, 102882, 109140, 220, 16, 8, 48909, 101276, 55675, 100303, 102829, 92911, 320, 100291, 100366, 100431, 320, 100497, 100287, 101755, 100431, 320, 61196, 100431, 86133, 101385, 100322, 24810, 101002, 101276, 102399, 44747, 101002, 102646, 102646, 122879, 100446, 102882, 102882, 100395, 48909, 45279, 101276, 55675, 100303, 102829, 92911, 1, 48909, 24810, 100291, 105869, 24810, 69258, 102882, 109140, 100518, 101411, 100277, 55675, 102348, 101263, 100282, 101263, 102889, 100915, 103630, 84736, 105781, 100273, 100428, 85410, 101207, 24810, 85410, 100436, 100866]

TRUE LABELS:
[45279, 101276, 55675, 100303, 102829, 92911, 320, 79468, 102882, 109140, 220, 17, 8, 100303, 101276, 55675, 100303, 102829, 92911, 111166, 106512, 79468, 100431, 101403, 100497, 100287, 101755, 100431, 103630, 79468, 100431, 86133, 101385, 100322, 101344, 100303, 121740, 102399, 44747, 101002, 101385, 102646, 122879, 69258, 101815, 102

In [31]:
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
pad_id = tokenizer.pad_token_id

labels = single_batch["labels"][0].detach().cpu()

print("eot_id:", eot_id)
print("pad_id:", pad_id)
print("EOT in labels?", (labels == eot_id).any().item())
print("EOT count:", (labels == eot_id).sum().item())
print("pad token in labels?", (labels == pad_id).any().item())
print("ignore count:", (labels == IGNORE_INDEX).sum().item())
print("last non-ignore labels:", labels[labels != IGNORE_INDEX][-10:].tolist())

eot_id: 128009
pad_id: 128004
EOT in labels? True
EOT count: 1
pad token in labels? False
ignore count: 100
last non-ignore labels: [84736, 105781, 100273, 100428, 85410, 101207, 24810, 85410, 100436, 128009]
